In [0]:
import pyspark.sql.functions as F

####Establish connection to the Datalake storage containers

In [0]:
spark.conf.set("fs.azure.account.auth.type.stinstacartlake.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.stinstacartlake.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.stinstacartlake.dfs.core.windows.net", "05b534b6-4a04-4d69-aa90-d1e381ef65f7")
spark.conf.set("fs.azure.account.oauth2.client.secret.stinstacartlake.dfs.core.windows.net", dbutils.secrets.get(scope="instacart-akv-scope", key="client-secret"))
spark.conf.set("fs.azure.account.oauth2.client.endpoint.stinstacartlake.dfs.core.windows.net", "https://login.microsoftonline.com/38c53c6c-2815-4635-9ecd-b14b896cf9a6/oauth2/token")

# Test the connection
# display(dbutils.fs.ls("abfss://bronze@stinstacartlake.dfs.core.windows.net/"))
# display(dbutils.fs.ls("abfss://silver@stinstacartlake.dfs.core.windows.net/"))

####Build user metrics based on current order and prior orders

In [0]:
# Read the silver orders table directly from storage
df_orders = spark.read.format("delta") \
    .load("abfss://silver@stinstacartlake.dfs.core.windows.net/orders")

# Read the silver order_products table directly from storage
df_order_products = spark.read.format("delta") \
    .load("abfss://silver@stinstacartlake.dfs.core.windows.net/order_products__prior")

# Join datasets and aggregate the columns to get the metrics
df_gld_user_metrics = df_orders.join(df_order_products, on="order_id", how="inner") \
    .groupBy("user_id") \
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.count("product_id").alias("total_items_purchased"),
        F.round(F.avg("days_since_prior_order"), 2).alias("avg_days_between_orders"),
        F.round(F.avg("order_hour_of_day"), 1).alias("avg_shopping_hour"),
        F.sum(F.col("reordered").cast("int")).alias("total_items_reordered")
    )

# Write the aggregated data to the Gold layer container as a Delta table
df_gld_user_metrics.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@stinstacartlake.dfs.core.windows.net/user_metrics")

# df_gld_user_metrics.head(10)

####Create the Unity catalog table in ADLS

In [0]:
%sql
CREATE EXTERNAL LOCATION instacart_gold_loc
URL 'abfss://gold@stinstacartlake.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL instacart_storage_cred);


In [0]:
# Register the table in Unity Catalog
spark.sql("""
    CREATE TABLE IF NOT EXISTS gold_user_metrics
    USING DELTA
    LOCATION 'abfss://gold@stinstacartlake.dfs.core.windows.net/user_metrics'
""")

# Preview the results
display(spark.sql("SELECT * FROM gold_user_metrics LIMIT 10"))